# chart.py Column Inspection

This notebook uses `chart.py` / `PointFigureChart` to build PnF charts from `data/nifty.csv` and print extracted columns from the chart matrix.

In [1]:
from csv import DictReader
from pathlib import Path
import sys
import types

root = Path.cwd()
if not (root / "chart.py").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

# chart.py imports tabulate only for pretty string output. Use a tiny fallback
# so column extraction works even when the optional package is not installed.
try:
    import tabulate  # noqa: F401
except ModuleNotFoundError:
    tabulate_module = types.ModuleType("tabulate")
    def simple_tabulate(rows, tablefmt="simple"):
        return "\n".join(" ".join(str(cell) for cell in row) for row in rows)
    tabulate_module.tabulate = simple_tabulate
    sys.modules["tabulate"] = tabulate_module

from chart import PointFigureChart

data_path = root / "data" / "nifty.csv"
data_path

WindowsPath('e:/Code_Projects/pnf/data/nifty.csv')

In [2]:
ts = {"date": [], "open": [], "high": [], "low": [], "close": []}
with data_path.open(newline="") as handle:
    for row in DictReader(handle):
        ts["date"].append(row["time"])
        ts["open"].append(float(row["open"]))
        ts["high"].append(float(row["high"]))
        ts["low"].append(float(row["low"]))
        ts["close"].append(float(row["close"]))

len(ts["close"]), ts["date"][:3], ts["date"][-3:]

(8659,
 ['1990-07-03', '1990-07-05', '1990-07-06'],
 ['2026-04-08', '2026-04-09', '2026-04-10'])

In [3]:
import numpy as np

def extract_chart_py_columns(pnf):
    columns = []
    matrix = pnf.matrix
    boxscale = pnf.boxscale
    action_index_matrix = pnf.action_index_matrix
    dates = pnf.ts.get("date")

    for column_index in range(matrix.shape[1]):
        box_indices = np.where(matrix[:, column_index] != 0)[0]
        if box_indices.size == 0:
            continue

        signs = matrix[box_indices, column_index]
        kind = "X" if signs[0] > 0 else "O"
        prices = boxscale[box_indices]
        action_indices = action_index_matrix[box_indices, column_index]
        action_indices = action_indices[action_indices >= 0]
        first_action_index = int(action_indices.min()) if action_indices.size else None
        last_action_index = int(action_indices.max()) if action_indices.size else None

        first_action_date = None
        last_action_date = None
        if dates is not None and first_action_index is not None and first_action_index < len(dates):
            first_action_date = str(dates[first_action_index])
        if dates is not None and last_action_index is not None and last_action_index < len(dates):
            last_action_date = str(dates[last_action_index])

        columns.append({
            "column": int(column_index),
            "type": kind,
            "box_count": int(box_indices.size),
            "low": float(np.min(prices)),
            "high": float(np.max(prices)),
            "start_box_index": int(box_indices[0]),
            "end_box_index": int(box_indices[-1]),
            "first_action_index": first_action_index,
            "last_action_index": last_action_index,
            "first_action_date": first_action_date,
            "last_action_date": last_action_date,
        })

    return columns

In [4]:
# chart.py log boxsize is in percent. 0.25 means 0.25%.
chart_close = PointFigureChart(
    ts=ts,
    method="cl",
    reversal=3,
    boxsize=0.25,
    scaling="log",
    title="Nifty chart.py close log",
)
close_columns = extract_chart_py_columns(chart_close)

len(close_columns), close_columns[:3], close_columns[-10:]

(1985,
 [{'column': 0,
   'type': 'X',
   'box_count': 25,
   'low': 278.97,
   'high': 296.2,
   'start_box_index': 20,
   'end_box_index': 44,
   'first_action_index': 1,
   'last_action_index': 6,
   'first_action_date': '1990-07-05',
   'last_action_date': '1990-07-12'},
  {'column': 1,
   'type': 'O',
   'box_count': 11,
   'low': 288.18,
   'high': 295.46,
   'start_box_index': 33,
   'end_box_index': 43,
   'first_action_index': 8,
   'last_action_index': 8,
   'first_action_date': '1990-07-16',
   'last_action_date': '1990-07-16'},
  {'column': 2,
   'type': 'X',
   'box_count': 74,
   'low': 288.9,
   'high': 346.66,
   'start_box_index': 34,
   'end_box_index': 107,
   'first_action_index': 9,
   'last_action_index': 16,
   'first_action_date': '1990-07-17',
   'last_action_date': '1990-07-30'}],
 [{'column': 1975,
   'type': 'O',
   'box_count': 11,
   'low': 24053.76,
   'high': 24661.92,
   'start_box_index': 1805,
   'end_box_index': 1815,
   'first_action_index': 8636,
 

In [5]:
chart_examples = {}
for label, method in (
    ("close", "cl"),
    ("high_low", "h/l"),
    ("high_low_close", "hlc"),
):
    pnf = PointFigureChart(
        ts=ts,
        method=method,
        reversal=3,
        boxsize=0.25,
        scaling="log",
        title=f"Nifty chart.py {label} log",
    )
    chart_examples[label] = {
        "chart": pnf,
        "columns": extract_chart_py_columns(pnf),
    }

[
    {
        "series": label,
        "chart_py_method": chart_examples[label]["chart"].method,
        "matrix_shape": chart_examples[label]["chart"].matrix.shape,
        "column_count": len(chart_examples[label]["columns"]),
        "last_column": chart_examples[label]["columns"][-1],
    }
    for label in chart_examples
]

[{'series': 'close',
  'chart_py_method': 'cl',
  'matrix_shape': (1862, 1985),
  'column_count': 1985,
  'last_column': {'column': 1984,
   'type': 'X',
   'box_count': 3,
   'low': 23874.26,
   'high': 23993.78,
   'start_box_index': 1802,
   'end_box_index': 1804,
   'first_action_index': 8658,
   'last_action_index': 8658,
   'first_action_date': '2026-04-10',
   'last_action_date': '2026-04-10'}},
 {'series': 'high_low',
  'chart_py_method': 'h/l',
  'matrix_shape': (1862, 3371),
  'column_count': 3371,
  'last_column': {'column': 3370,
   'type': 'X',
   'box_count': 6,
   'low': 23755.33,
   'high': 24053.76,
   'start_box_index': 1800,
   'end_box_index': 1805,
   'first_action_index': 8658,
   'last_action_index': 8658,
   'first_action_date': '2026-04-10',
   'last_action_date': '2026-04-10'}},
 {'series': 'high_low_close',
  'chart_py_method': 'hlc',
  'matrix_shape': (1862, 2451),
  'column_count': 2451,
  'last_column': {'column': 2450,
   'type': 'X',
   'box_count': 6,
 

In [6]:
for label, item in chart_examples.items():
    print("\n", label)
    for column in item["columns"][-5:]:
        print(column)


 close
{'column': 1980, 'type': 'X', 'box_count': 13, 'low': 22598.18, 'high': 23285.53, 'start_box_index': 1780, 'end_box_index': 1792, 'first_action_index': 8648, 'last_action_index': 8649, 'first_action_date': '2026-03-24', 'last_action_date': '2026-03-25'}
{'column': 1981, 'type': 'O', 'box_count': 16, 'low': 22373.6, 'high': 23227.46, 'start_box_index': 1776, 'end_box_index': 1791, 'first_action_index': 8650, 'last_action_index': 8651, 'first_action_date': '2026-03-27', 'last_action_date': '2026-03-30'}
{'column': 1982, 'type': 'X', 'box_count': 28, 'low': 22429.54, 'high': 23993.78, 'start_box_index': 1777, 'end_box_index': 1804, 'first_action_index': 8652, 'last_action_index': 8656, 'first_action_date': '2026-04-01', 'last_action_date': '2026-04-08'}
{'column': 1983, 'type': 'O', 'box_count': 3, 'low': 23814.72, 'high': 23933.94, 'start_box_index': 1801, 'end_box_index': 1803, 'first_action_index': 8657, 'last_action_index': 8657, 'first_action_date': '2026-04-09', 'last_action

In [7]:
# chart.py also has its own ASCII-like printer via __str__.
print(chart_close)

Point & Figure (log|cl) 0.25% x 3 | Nifty chart.py close log
26316.05 . . . . . . . . . X . . . . . . . . . . . . . . . . . . . . 26316.05
26250.43 . . . . . . . . . X O . . . . . . . . . . . . . . . . . . . 26250.43
26184.96 . . . X . X . X . X O . . . . . . . . . . . . . . . . . . . 26184.96
26119.66 . . . X O X O X O X O . . . . . . . . . . . . . . . . . . . 26119.66
26054.53 . . . X O X O X O X O . . . . . . . . . . . . . . . . . . . 26054.53
25989.55 . X . X O X O . O X O . . . . . . . . . . . . . . . . . . . 25989.55
25924.74 . X O X O . . . O X O . . X . . . . . . . . . . . . . . . . 25924.74
25860.09 . X O X . . . . O X O . . X O . . . . . . . . . . . . . . . 25860.09
25795.6 . X O X . . . . O . O . . X O X . . . . . . . . . . . . . . 25795.6
25731.27 . X O X . . . . . . O . . X O X O . . . . . . . . . . . . . 25731.27
25667.11 . X O X . . . . . . O . . X O X O X . . . . . . . . . . . . 25667.11
25603.1 . X O X . . . . . . O . . X O X O X O . . . . . . . . . . . 25603.1
25539.2